# 데이터 수집

이건 이미 완료해서, 노션 속 데이터 쓰면 됩니당

# 데이터 전처리

In [ ]:
import re
!pip install konlpy
from konlpy.tag import Okt
import pandas as pd

okt = Okt()

def clean_text(text):
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# 부정어 도출 안하는거
def extract_keywords(text):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r'[^\w\s가-힣]', '', text)
    morphs = okt.pos(text, norm=True, stem=True)
    result = []

    for word, tag in morphs:
        if tag in ['Noun', 'Verb', 'Adjective']:
            result.append(word)

    return result

In [ ]:
# 부정어 도출할거면 이거 쓰면 됩니당
def extract_keywords_with_negation(text):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r'[^\w\s가-힣]', '', text)
    morphs = okt.pos(text, norm=True, stem=True)
    result = []
    i = 0

    while i < len(morphs):
        word, tag = morphs[i]

        # Case 1: '지 않다' or '지 못하다'
        if word == '지' and i + 1 < len(morphs):
            next_word, next_tag = morphs[i + 1]
            if next_word in ['않다', '못하다'] and next_tag == 'Verb':
                if result:
                    prev = result.pop()
                    result.append(f'NOT_{prev}')
                i += 2
                continue

        # Case 2: '않다', '못하다', '없다' (보조용언/형용사 → 앞 단어 부정)
        if word in ['않다', '못하다', '없다'] and tag in ['Verb', 'Adjective']:
            if result:
                prev = result.pop()
                result.append(f'NOT_{prev}')
            i += 1
            continue

        # ✅ Case 3: 부정 부사 (안, 못, 아니)
        if word in ['안', '못', '아니'] and tag in ['Adverb', 'Noun']:
            if i + 1 < len(morphs):
                next_word, next_tag = morphs[i + 1]

                if next_tag in ['Verb', 'Adjective']:
                    # 👇 수정된 부분: '나다/보이다/들리다'는 앞 명사 부정
                    if result and result[-1] and next_word in ['나다', '보이다', '들리다']:
                        prev = result[-1]  # pop하지 않고 유지
                        result.append(f'NOT_{next_word}')  # 동사에 부정
                    else:
                        result.append(f'NOT_{next_word}')
                    i += 2
                    continue


        # 일반 키워드: 명사/동사/형용사
        if tag in ['Noun', 'Verb', 'Adjective']:
            result.append(word)

        i += 1

    return result

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 폴더 경로만 주의해서 지정하면 됩니당
import os
import pandas as pd

# 폴더 경로 설정
folder_path = '/content/drive/MyDrive/김치찌개리뷰'

# 폴더 내 모든 파일 불러오기
file_names = os.listdir(folder_path)

# 각 파일 반복 처리
for i in file_names:
    if i.startswith('구분X_'):  # 이미 처리된 파일은 건너뛰기
        continue

    file_path = os.path.join(folder_path, i)
    print(f"처리 중: {file_path}")

    # CSV 읽기
    df = pd.read_csv(file_path)

    # 텍스트 전처리 및 키워드 추출
    df['token'] = df['리뷰'].apply(clean_text).apply(extract_keywords_with_negation)

    # 결과 저장
    save_path = os.path.join(folder_path, '구분X_' + i)
    df.to_csv(save_path, index=False)
    print(f"저장 완료: {save_path}")

In [ ]:
df_kimchi_1 = pd.read_csv('/content/drive/MyDrive/김치찌개리뷰/구분X_금돼지식당_리뷰_250602.csv')
df_kimchi_2 = pd.read_csv('/content/drive/MyDrive/김치찌개리뷰/구분X_명장김치찌개_리뷰_250602.csv')
df_kimchi_3 = pd.read_csv('/content/drive/MyDrive/김치찌개리뷰/구분X_신사강김치찌개_리뷰_250602.csv')

In [ ]:
# 중복 토큰 제거
def remove_duplicated_token(tokens):
    seen = set()
    result = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            result.append(token)
    return result


# 파일로 저장후 다시 불러왔을 때, 리스트 형태가 아닌 str로 표시되는 문제 해결
import ast
def clean_dataframe(df):
    df['token'] = df['token'].apply(ast.literal_eval)
    df['token'] = df['token'].apply(remove_duplicated_token)
    return df.drop_duplicates(subset='리뷰', keep='first')

In [ ]:
df_kimchi_1 = clean_dataframe(df_kimchi_1)  # 금돼지식당
df_kimchi_2 = clean_dataframe(df_kimchi_2)  # 명장김치찌개
df_kimchi_3 = clean_dataframe(df_kimchi_3)  # 신사강김치찌개

# 빈도수 구하기

In [ ]:
from itertools import chain

all_tokens_kimchi = list(chain.from_iterable(
    df['token'] for df in [df_kimchi_1, df_kimchi_2, df_kimchi_3]
))

In [ ]:
from collections import Counter
from itertools import chain

# 구조 확실하게 점검 및 flatten
all_tokens_kimchi = list(chain.from_iterable(
    token for df in [df_kimchi_1, df_kimchi_2, df_kimchi_3] for token in df['token']
))

# 확인 (문자열인지 체크)
assert all(isinstance(tok, str) for tok in all_tokens_kimchi), "토큰이 문자열이 아닙니다"

# Counter 적용
token_freq_kimchi = Counter(all_tokens_kimchi)

# DataFrame으로 보기 좋게 변환
freq_df_kimchi = pd.DataFrame(token_freq_kimchi.items(), columns=['토큰', '빈도수']).sort_values(by='빈도수', ascending=False)
freq_df_kimchi

In [ ]:
# 엑셀로 보는게 편해서.. 이거 보면서 유의미한 맛 표현 키워드 도출
freq_df_kimchi.to_excel('/content/drive/MyDrive/김치찌개리뷰/구분X_토큰_빈도수.xlsx', index=False)

In [ ]:
from collections import Counter
from itertools import chain
import numpy as np
import pandas as pd

# 1. 맛 관련 키워드 리스트 (개별 단어)
kimchi_keywords = [
    '짜다', '시원하다', '든든하다', '진하다', '건더기',
    '칼칼하다', '얼큰하다', '매콤', '부드럽다', '달다',
    '신맛', '싱겁다', '개운하다', '담백하다',
    '시큼하다', '밍밍하다', '큼직하다'
]

# 2. 맛 그룹 딕셔너리 (필요한 단어만 그룹화, 예: '맑다'에 '맑다', '맑은' 묶기)
taste_groups = {
    '짜다': ['짜다', '짭잘하다', '짭짤하다'],
    '맵다': ['맵다', '매콤', '얼큰하다']
}



# 그룹 키워드 추출
group_keywords = set()
for v in taste_groups.values():
    group_keywords.update(v)
group_names = list(taste_groups.keys())

# 개별 키워드 추출
individual_keywords = [kw for kw in kimchi_keywords if kw not in group_keywords]

In [ ]:
total_keywords = individual_keywords + group_names

# 1. 컬럼명 생성
keyword_columns = [f"{kw}_비율" for kw in total_keywords]
base_columns = ['가게명', '리뷰수','키워드포함_리뷰수',  '평균평점', '맛있다_수', '맛있다_비율']
total_info_df = pd.DataFrame(columns=base_columns + keyword_columns)

# 2. 김치찌개 가게 데이터 반복 처리
for temp in [df_kimchi_1, df_kimchi_2, df_kimchi_3]:
    review_count = len(temp)
    temp = temp[temp['token'].apply(lambda tokens: any(token in tokens for token in total_keywords))]
    token_freq = Counter(chain.from_iterable(temp['token']))
    df_freq = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수'])

    review_count_keyword = len(temp)

    try:
        option = temp['세부옵션'].value_counts().idxmax()
    except:
        option = np.nan

    mean_score = temp['별점'].mean()
    delicious_count = df_freq[df_freq['토큰'] == '맛있다'].values[0][1] if '맛있다' in df_freq['토큰'].values else 0
    delicious_ratio = delicious_count / review_count_keyword if review_count_keyword > 0 else 0

    # 3. 개별 키워드 비율
    keyword_ratios = []
    for kw in individual_keywords:
        kw_count = temp['token'].apply(lambda tokens: kw in tokens).sum()
        kw_ratio = kw_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(kw_ratio)

    # 4. 그룹 키워드 비율
    for g in group_names:
        group_kw_list = taste_groups[g]
        group_count = temp['token'].apply(lambda tokens: any(kw in tokens for kw in group_kw_list)).sum()
        group_ratio = group_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(group_ratio)

    total_info_df.loc[len(total_info_df)] = [np.nan, review_count, review_count_keyword, mean_score,
                                             delicious_count, delicious_ratio] + keyword_ratios

# 5. 가게명 지정 및 정리
total_info_df['가게명'] = ['금돼지식당', '명장김치찌개', '신사강김치찌개']
total_info_df = total_info_df.T
total_info_df.columns = total_info_df.iloc[0]
total_info_df = total_info_df[1:]
total_info_df = total_info_df.loc[~(total_info_df == 0).all(axis=1)]

total_info_df

In [ ]:
# 카테고리별 키워드 사전 정의
flavor_categories_kimchi = {
    "맑고 깔끔한 맛": ["시원하다", "개운하다", "담백하다", "밍밍하다"],
    "매운맛/자극": ["칼칼하다", "얼큰하다", "매콤", "신맛", "시큼하다"],
    "간 관련": ["짜다", "싱겁다"],
    "푸짐함/양": ["든든하다", "건더기", "큼직하다"],
    "진한 맛": ["진하다"],
    "식감": ["부드럽다"],
    "단맛": ["달다"]
}
def label_flavor_category(index_word):
    for category, keywords in flavor_categories_kimchi.items():
        for keyword in keywords:
            if keyword in index_word:
                return category
    return "기타"

total_info_df["범주"] = total_info_df.index.map(label_flavor_category)

In [ ]:
total_info_df['범주'].unique()

In [ ]:
total_info_df[total_info_df['범주'] == '식감']

In [ ]:
total_info_df.to_csv('/content/drive/MyDrive/김치찌개리뷰/맛_키워드_비율_전체.csv', encoding='utf-8-sig')

In [ ]:
df = total_info_df.copy()

In [ ]:
# 5번째 행부터 끝까지 숫자 데이터만 추출해서 합계 계산
numeric_sum = df.iloc[5:, :-1].sum()

# 결과 출력
print(numeric_sum)

맛 표현이 독립적이진 않다보니 다 더했을 때 1인건 아님

ex) 깔끔하고 넉넉해요~~ 의 경우 '깔끔하다'와 '넉넉하다'두 표현이 들어있음

In [ ]:
# 범주 컬럼 기준으로 그룹화한 뒤, 숫자형 컬럼에 대해 합계 계산
grouped_df = df.iloc[5:].groupby('범주').sum()

# 결과 확인
print(grouped_df)

In [ ]:
# 1. 나눔글꼴 설치 및 설정
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import os

# 설치된 나눔글꼴을 직접 등록
font_dirs = ['/usr/share/fonts/truetype/nanum']
font_files = fm.findSystemFonts(fontpaths=font_dirs)
for font_file in font_files:
    fm.fontManager.addfont(font_file)

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 2. 방사형 차트 기본 설정
categories = grouped_df.index.tolist()
num_vars = len(categories)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

colors = ['#FF8C00', '#DC143C', '#4169E1', '#228B22', '#8A2BE2', '#FF1493']
brands = grouped_df.columns.tolist()

# 3. y축 최대값 계산 (모든 값 중 가장 큰 값)
max_val = grouped_df.values.max()

# 4. Subplot 생성 (3행 2열)
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 14), subplot_kw=dict(polar=True))
axes = axes.flatten()

for idx, brand in enumerate(brands):
    values = grouped_df[brand].tolist()
    values += values[:1]

    ax = axes[idx]
    ax.plot(angles, values, color=colors[idx], linewidth=2)
    ax.fill(angles, values, color=colors[idx], alpha=0.25)

    # 축 설정
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), categories)
    ax.set_ylim(0, max_val)  # 모든 그래프에 동일한 y축 범위 적용
    ax.set_title(f"{brand}의 맛 범주 비율", y=1.1)
    ax.grid(True)

# 남는 subplot 제거
for j in range(len(brands), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
